In [1]:
import geopandas as gpd
import folium
print("GeoPandas:", gpd.__version__)
print("Folium:", folium.__version__)

GeoPandas: 1.1.3
Folium: 0.20.0


In [2]:
import subprocess
subprocess.run(['pip', 'install', 'folium'])

CompletedProcess(args=['pip', 'install', 'folium'], returncode=0)

In [3]:
import geopandas as gpd
import folium
print("GeoPandas:", gpd.__version__)
print("Folium:", folium.__version__)

GeoPandas: 1.1.3
Folium: 0.20.0


In [4]:
import pandas as pd
import folium
import geopandas as gpd
import json

# Load your master solar dataset
df = pd.read_csv('../data/west_africa_solar_data_2014_2025.csv')

# Calculate annual average GHI per city
city_ghi = df.groupby(['City', 'Country', 'Climate_Zone'])['GHI'].mean().reset_index()
city_ghi.columns = ['City', 'Country', 'Climate_Zone', 'Mean_GHI']
city_ghi['Mean_GHI'] = city_ghi['Mean_GHI'].round(3)

# Add coordinates
coords = {
    'Lagos':        (6.5244,   3.3792),
    'Abuja':        (9.0765,   7.3986),
    'Kano':         (12.0022,  8.5920),
    'Accra':        (5.6037,  -0.1870),
    'Dakar':        (14.7167, -17.4677),
    'Abidjan':      (5.3600,  -4.0083),
    'Bamako':       (12.6392,  -8.0029),
    'Ouagadougou':  (12.3647,  -1.5332),
    'Niamey':       (13.5137,   2.1098),
    'Conakry':      (9.6412,  -13.5784),
    'Lome':         (6.1375,   1.2123),
    'Cotonou':      (6.3654,   2.4183),
    'Freetown':     (8.4657,  -13.2317),
    'Monrovia':     (6.2907,  -10.7605),
    'Nouakchott':   (18.0735, -15.9582),
    'Banjul':       (13.4549, -16.5790),
    'Bissau':       (11.8636, -15.5977),
    'Praia':        (14.9331, -23.5133),
}

city_ghi['lat'] = city_ghi['City'].map(lambda c: coords[c][0])
city_ghi['lon'] = city_ghi['City'].map(lambda c: coords[c][1])

print(f"Cities loaded: {len(city_ghi)}")
print(city_ghi[['City', 'Country', 'Climate_Zone', 'Mean_GHI']].to_string(index=False))

Cities loaded: 18
       City       Country  Climate_Zone  Mean_GHI
    Abidjan Cote d'Ivoire Coastal Humid     4.630
      Abuja       Nigeria       Savanna     5.269
      Accra         Ghana Coastal Humid     4.884
     Bamako          Mali         Sahel     5.805
     Banjul        Gambia         Sahel     5.776
     Bissau Guinea-Bissau Coastal Humid     5.503
    Conakry        Guinea Coastal Humid     5.273
    Cotonou         Benin Coastal Humid     4.642
      Dakar       Senegal         Sahel     5.794
   Freetown  Sierra Leone Coastal Humid     4.957
       Kano       Nigeria         Sahel     5.973
      Lagos       Nigeria Coastal Humid     4.592
       Lome          Togo Coastal Humid     4.837
   Monrovia       Liberia Coastal Humid     4.567
     Niamey         Niger         Sahel     6.052
 Nouakchott    Mauritania   Desert/Arid     6.291
Ouagadougou  Burkina Faso         Sahel     5.822
      Praia    Cape Verde   Desert/Arid     5.889


In [5]:
# ================================================
# INTERACTIVE FOLIUM MAP — West Africa Solar GHI
# ================================================

# Colour scheme by climate zone
zone_colors = {
    'Coastal Humid': '#0099ff',
    'Savanna':       '#00c27c',
    'Sahel':         '#f7c948',
    'Desert/Arid':   '#ff6b35'
}

# Create base map centered on West Africa
m = folium.Map(
    location=[12, -5],
    zoom_start=5,
    tiles='CartoDB positron'
)

# Add city markers with popups
for _, row in city_ghi.iterrows():
    color = zone_colors[row['Climate_Zone']]

    # Circle size based on GHI value
    radius = (row['Mean_GHI'] - 4.0) * 8

    # Popup content
    popup_html = f"""
    <div style="font-family: Arial; width: 200px;">
        <h4 style="color: {color}; margin-bottom: 5px;">
            {row['City']}, {row['Country']}
        </h4>
        <table style="width:100%; font-size:13px;">
            <tr>
                <td><b>Annual GHI</b></td>
                <td>{row['Mean_GHI']:.3f} kWh/m²/day</td>
            </tr>
            <tr>
                <td><b>Climate Zone</b></td>
                <td>{row['Climate_Zone']}</td>
            </tr>
            <tr>
                <td><b>Solar Rank</b></td>
                <td>#{int(city_ghi['Mean_GHI'].rank(ascending=False)[_].item())} of 18</td>
            </tr>
        </table>
    </div>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"{row['City']} — {row['Mean_GHI']:.2f} kWh/m²/day"
    ).add_to(m)

    # City label
    folium.Marker(
        location=[row['lat'], row['lon']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px; font-weight:bold; '
                 f'color:#333; white-space:nowrap;">{row["City"]}</div>',
            icon_size=(80, 20),
            icon_anchor=(0, 0)
        )
    ).add_to(m)

# Add legend
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 15px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial; font-size: 13px;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.2);">
    <b style="font-size:14px;">Climate Zone</b><br><br>
    <span style="color:#ff6b35;">●</span> Desert/Arid<br>
    <span style="color:#f7c948;">●</span> Sahel<br>
    <span style="color:#00c27c;">●</span> Savanna<br>
    <span style="color:#0099ff;">●</span> Coastal Humid<br><br>
    <b style="font-size:14px;">Circle Size</b><br>
    <span style="font-size:11px; color:#666;">Proportional to annual GHI</span>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Save as HTML
m.save('../reports/west_africa_solar_interactive.html')
print("Interactive map saved!")
print("Open: africa-energy-analytics/reports/west_africa_solar_interactive.html")

Interactive map saved!
Open: africa-energy-analytics/reports/west_africa_solar_interactive.html


In [6]:
# ================================================
# CHOROPLETH MAP — Electricity Access + Solar GHI
# ================================================

# Load World Bank electricity access data
wb_df = pd.read_csv('../data/API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_127016.csv',
                    skiprows=4)

# Map country names to match shapefile
country_name_map = {
    'Nigeria':       'Nigeria',
    'Ghana':         'Ghana',
    'Senegal':       'Senegal',
    "Cote d'Ivoire": "Cote d'Ivoire",
    'Mali':          'Mali',
    'Burkina Faso':  'Burkina Faso',
    'Niger':         'Niger',
    'Guinea':        'Guinea',
    'Togo':          'Togo',
    'Benin':         'Benin',
    'Sierra Leone':  'Sierra Leone',
    'Liberia':       'Liberia',
    'Mauritania':    'Mauritania',
    'Gambia':        'Gambia, The',
    'Guinea-Bissau': 'Guinea-Bissau',
    'Cape Verde':    'Cabo Verde',
}

# Filter World Bank data
wa_access = wb_df[wb_df['Country Name'].isin(country_name_map.values())]
wa_access = wa_access[['Country Name', '2023']].dropna()
wa_access.columns = ['Country', 'Electricity_Access_2023']

# Clean display names
display_map = {v: k for k, v in country_name_map.items()}
wa_access['Display_Name'] = wa_access['Country'].map(display_map)

print(wa_access[['Display_Name', 'Electricity_Access_2023']].to_string(index=False))

 Display_Name  Electricity_Access_2023
        Benin                     57.0
 Burkina Faso                     21.7
Cote d'Ivoire                     72.4
   Cape Verde                     98.6
        Ghana                     89.5
       Guinea                     51.1
       Gambia                     66.9
Guinea-Bissau                     40.5
      Liberia                     32.5
         Mali                     54.5
   Mauritania                     50.3
        Niger                     20.1
      Nigeria                     61.2
      Senegal                     74.2
 Sierra Leone                     35.5
         Togo                     59.2


In [7]:
# ================================================
# CHOROPLETH MAP — Electricity Access by Country
# + Solar GHI circles overlay
# ================================================

# Load shapefile
world = gpd.read_file('../data/countries.geojson')

# Filter West Africa
wa_countries = [
    'Nigeria', 'Ghana', 'Senegal', "Côte d'Ivoire", 'Mali',
    'Burkina Faso', 'Niger', 'Guinea', 'Togo', 'Benin',
    'Sierra Leone', 'Liberia', 'Mauritania', 'Gambia',
    'Guinea-Bissau', 'Cabo Verde'
]
wa_map = world[world['NAME'].isin(wa_countries)].copy()

# Match names for merge
name_fix = {
    "Côte d'Ivoire": "Cote d'Ivoire",
    'Cabo Verde':    'Cape Verde',
    'Gambia':        'Gambia',
}
wa_map['merge_name'] = wa_map['NAME'].replace(name_fix)

# Merge electricity access data
wa_merged = wa_map.merge(
    wa_access[['Display_Name', 'Electricity_Access_2023']],
    left_on='merge_name',
    right_on='Display_Name',
    how='left'
)

print(f"Countries in map: {len(wa_merged)}")
print(wa_merged[['NAME', 'Electricity_Access_2023']].to_string(index=False))

Countries in map: 16
         NAME  Electricity_Access_2023
         Togo                     59.2
 Sierra Leone                     35.5
      Senegal                     74.2
      Nigeria                     61.2
        Niger                     20.1
   Mauritania                     50.3
         Mali                     54.5
      Liberia                     32.5
Guinea-Bissau                     40.5
       Guinea                     51.1
        Ghana                     89.5
       Gambia                     66.9
Côte d'Ivoire                     72.4
   Cabo Verde                     98.6
 Burkina Faso                     21.7
        Benin                     57.0


In [8]:
# ================================================
# BUILD CHOROPLETH MAP
# ================================================

# Convert to JSON for Folium
wa_json = wa_merged.to_json()

# Create base map
m2 = folium.Map(
    location=[12, -5],
    zoom_start=5,
    tiles='CartoDB positron'
)

# Add choropleth layer — electricity access
folium.Choropleth(
    geo_data=wa_json,
    name='Electricity Access',
    data=wa_merged,
    columns=['NAME', 'Electricity_Access_2023'],
    key_on='feature.properties.NAME',
    fill_color='RdYlGn',
    fill_opacity=0.7,
    line_opacity=0.5,
    legend_name='Electricity Access (% of population, 2023)',
    nan_fill_color='lightgray',
).add_to(m2)

# Add tooltip on country hover
folium.GeoJson(
    wa_json,
    name='Country Info',
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': 'transparent',
        'weight': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['NAME'],
        aliases=['Country:'],
        style="font-family: Arial; font-size: 13px;"
    )
).add_to(m2)

# Add solar GHI circles on top
for _, row in city_ghi.iterrows():
    color = zone_colors[row['Climate_Zone']]
    radius = (row['Mean_GHI'] - 4.0) * 8

    popup_html = f"""
    <div style="font-family: Arial; width: 210px;">
        <h4 style="color: {color}; margin-bottom: 5px;">
            {row['City']}, {row['Country']}
        </h4>
        <table style="width:100%; font-size:13px;">
            <tr><td><b>Annual GHI</b></td>
                <td>{row['Mean_GHI']:.3f} kWh/m²/day</td></tr>
            <tr><td><b>Climate Zone</b></td>
                <td>{row['Climate_Zone']}</td></tr>
        </table>
    </div>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=220),
        tooltip=f"{row['City']} — GHI: {row['Mean_GHI']:.2f} kWh/m²/day"
    ).add_to(m2)

# Add legend
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 15px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial; font-size: 13px;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.2);">
    <b style="font-size:14px;">Solar GHI — Climate Zone</b><br><br>
    <span style="color:#ff6b35;">●</span> Desert/Arid<br>
    <span style="color:#f7c948;">●</span> Sahel<br>
    <span style="color:#00c27c;">●</span> Savanna<br>
    <span style="color:#0099ff;">●</span> Coastal Humid<br><br>
    <b style="font-size:12px; color:#666;">Circle size ∝ annual GHI</b><br><br>
    <b style="font-size:14px;">Country fill</b><br>
    <span style="font-size:11px; color:#666;">Electricity access % (2023)</span><br>
    <span style="color:#d73027;">■</span> Low access<br>
    <span style="color:#fee08b;">■</span> Medium access<br>
    <span style="color:#1a9850;">■</span> High access<br>
</div>
"""
m2.get_root().html.add_child(folium.Element(legend_html))

# Layer control
folium.LayerControl().add_to(m2)

# Save
m2.save('../reports/west_africa_choropleth.html')
print("Choropleth map saved!")

Choropleth map saved!
